<a href="https://colab.research.google.com/github/inidmoremoney/NLP2/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas

In [ ]:
%%writefile train_baseline.py
import argparse
import os

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

LABEL_NAMES = ["부정", "긍정"]  # index 0 = 부정, 1 = 긍정


# ──────────────────────────────────────────────────────────────
# 1. 인자 파싱
# ──────────────────────────────────────────────────────────────
def get_args():
    p = argparse.ArgumentParser(description="KoELECTRA 식당 리뷰 감성분류 baseline")

    # 데이터 / 모델
    p.add_argument("--train_file", type=str, default="train.csv")
    p.add_argument("--val_file", type=str, default="val.csv")
    p.add_argument("--test_file", type=str, default="test.csv")
    p.add_argument("--model_name", type=str, default="daekeun-ml/koelectra-small-v3-nsmc")
    p.add_argument("--output_dir", type=str, default="./koelectra-restaurant")

    # 하이퍼파라미터 (값들을 바꿔가며 validation 성능을 비교해 볼 것)
    p.add_argument("--lr", type=float, default=3e-5)
    p.add_argument("--batch_size", type=int, default=75)
    p.add_argument("--epochs", type=int, default=20)
    p.add_argument("--max_len", type=int, default=150)
    p.add_argument("--patience", type=int, default=3, help="EarlyStopping patience (epoch 단위)")
    p.add_argument("--seed", type=int, default=42)

    # 예측 모드
    p.add_argument("--predict", type=str, default=None,
                   help="학습 대신 이 문장을 예측 (output_dir 에 학습된 모델 필요)")
    return p.parse_args()


# ──────────────────────────────────────────────────────────────
# 2. 평가 지표
# ──────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def show(metrics, prefix):
    for k, v in metrics.items():
        if k.startswith(prefix) and isinstance(v, float):
            print(f"  {k:18s}: {v:.4f}")


# ──────────────────────────────────────────────────────────────
# 3. 학습 + 평가
# ──────────────────────────────────────────────────────────────
def train(args):
    set_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INFO] device = {device}")

    # 3-1. 토크나이저 / 모델 로드
    #   NSMC 로 이미 2-class 학습된 모델이라 분류 헤드를 그대로 재사용
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        args.model_name,
        num_labels=2,
        id2label={0: LABEL_NAMES[0], 1: LABEL_NAMES[1]},
        label2id={LABEL_NAMES[0]: 0, LABEL_NAMES[1]: 1},
    )

    # 3-2. 데이터 로드
    raw = load_dataset(
        "csv",
        data_files={
            "train": args.train_file,
            "validation": args.val_file,
            "test": args.test_file,
        },
        encoding="utf-8-sig",
    )
    print(raw)

    # 결측 행 제거
    raw = raw.filter(lambda x: x["Sentence"] is not None and x["Label"] is not None)

    # 3-3. 토크나이징
    def tokenize(batch):
        return tokenizer(batch["Sentence"], truncation=True, max_length=args.max_len)

    ds = raw.map(tokenize, batched=True, remove_columns=["Sentence"])

    ds = ds.rename_column("Label", "labels")
    collator = DataCollatorWithPadding(tokenizer)

    # 3-4. 학습 설정
    training_args = TrainingArguments(
        output_dir=args.output_dir,
        learning_rate=args.lr,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        num_train_epochs=args.epochs,
        weight_decay=0.01,
        warmup_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=20,
        fp16=(device == "cuda"),
        report_to="none",
        seed=args.seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"],
        eval_dataset=ds["validation"],    # epoch 마다 validation 으로 평가
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.patience)],
    )

    # 3-5. 학습 전 성능 (zero-shot baseline)
    #   NSMC 영화리뷰 모델을 식당 리뷰에 그대로 적용했을 때 점수.
    #   파인튜닝 후 test 점수와 비교하는 것이 핵심.
    print("\n[Before fine-tuning] 영화리뷰 모델을 식당리뷰에 그대로 적용 (test)")
    show(trainer.evaluate(ds["test"], metric_key_prefix="test"), prefix="test_")

    # 3-6. 학습 (validation 기준으로 best epoch 선택)
    print("\n[Training]")
    trainer.train()

    # 3-7. 학습 후 성능
    print("\n[After fine-tuning] validation (best epoch)")
    show(trainer.evaluate(ds["validation"], metric_key_prefix="val"), prefix="val_")

    print("\n[After fine-tuning] test  ← 최종 보고용 점수")
    show(trainer.evaluate(ds["test"], metric_key_prefix="test"), prefix="test_")

    # 3-8. 최종 모델 저장
    final_dir = os.path.join(args.output_dir, "final")
    trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)
    print(f"\n[INFO] 모델 저장 완료 → {final_dir}")


# ──────────────────────────────────────────────────────────────
# 4. 예측
# ──────────────────────────────────────────────────────────────
@torch.no_grad()
def predict(args):
    final_dir = os.path.join(args.output_dir, "final")
    if not os.path.isdir(final_dir):
        raise FileNotFoundError(f"{final_dir} 가 없습니다. 먼저 학습을 실행하세요.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(final_dir)
    model = AutoModelForSequenceClassification.from_pretrained(final_dir).to(device).eval()

    enc = tokenizer(args.predict, truncation=True, max_length=args.max_len,
                    return_tensors="pt").to(device)
    probs = torch.softmax(model(**enc).logits, dim=-1)[0]
    idx = int(probs.argmax())

    print(f"입력 : {args.predict}")
    print(f"예측 : {LABEL_NAMES[idx]}  (확률 {probs[idx].item():.3f})")
    print(f"      부정 {probs[0].item():.3f} / 긍정 {probs[1].item():.3f}")


# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    args = get_args()
    if args.predict:
        predict(args)
    else:
        train(args)


Overwriting train_baseline.py


In [ ]:
!python train_baseline.py --patience 100 --epochs 20 --output_dir ./run_nostop

[INFO] device = cuda
Loading weights: 100% 203/203 [00:00<00:00, 5976.64it/s]
DatasetDict({
    train: Dataset({
        features: ['Sentence', 'Label'],
        num_rows: 1840
    })
    validation: Dataset({
        features: ['Sentence', 'Label'],
        num_rows: 230
    })
    test: Dataset({
        features: ['Sentence', 'Label'],
        num_rows: 230
    })
})
Map: 100% 1840/1840 [00:00<00:00, 18306.01 examples/s]
Map: 100% 230/230 [00:00<00:00, 15872.61 examples/s]
Map: 100% 230/230 [00:00<00:00, 15221.93 examples/s]

[Before fine-tuning] 영화리뷰 모델을 식당리뷰에 그대로 적용 (test)
  0% 0/4 [00:00<?, ?it/s][transformers] early stopping required metric_for_best_model, but did not find eval_f1 so early stopping is disabled
100% 4/4 [00:00<00:00, 57.20it/s]
  test_loss         : 0.7244
  test_model_preparation_time: 0.0031
  test_accuracy     : 0.8174
  test_precision    : 0.9200
  test_recall       : 0.7823
  test_f1           : 0.8456
  test_runtime      : 0.3997
  test_samples_per_second: 

In [ ]:
!python analyze_errors.py --model_dir ./run_nostop/final

[INFO] device = cuda, model = ./run_nostop/final
Loading weights: 100% 203/203 [00:00<00:00, 5241.62it/s]

[val]  230문장
accuracy 0.8913  (205/230)
max_len=150 토큰을 넘어 잘리는 문장: 0개

              precision    recall  f1-score   support

          부정     0.8919    0.7952    0.8408        83
          긍정     0.8910    0.9456    0.9175       147

    accuracy                         0.8913       230
   macro avg     0.8915    0.8704    0.8791       230
weighted avg     0.8913    0.8913    0.8898       230

혼동행렬 (행 = 정답, 열 = 예측)
             예측:부정  예측:긍정
  정답:부정        66        17
  정답:긍정         8       139

길이 구간별 오류율
         문장수  오답수    오류율
len_bin                 
~10자      44    6  0.136
11~20자    80    6  0.075
21~40자    67    9  0.134
41~80자    35    4  0.114
81자~       4    0  0.000

틀린 문장 25개 → ./analysis/errors_val.csv
가장 확신하며 틀린 5개:
  정답 부정 / 예측 긍정 (확신 1.00)  식감도 그렇고하
  정답 부정 / 예측 긍정 (확신 1.00)  누린내 많이 나는 돼지국밥 잘 드시면 강추입니다
  정답 긍정 / 예측 부정 (확신 1.00)  그냥 평범
  정답 부정 / 예측 긍정 (확신 1.00)  